## MULTI DATASET PREPROCESSING

In [1]:
import re
import string

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [2]:
# Load multilingual dataset
df_multi = pd.read_csv("../../data/raw/multilingual_dataset.csv")
# Normalize language codes
df_multi['language'] = df_multi['language'].astype(str).str.upper()
print(df_multi[['language']].value_counts().head(10).to_string())

language      
ARABIC            900
DARIJA_ARABIZI    900
DARIJA_ARABIC     900
ENGLISH           900
FRENCH            900


In [3]:
# Download NLTK resources (once)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [4]:
# Stopwords and lemmatizer
stop_words_en = set(stopwords.words('english'))
stop_words_fr = set(stopwords.words('french'))

try:
    stop_words_ar = set(stopwords.words('arabic'))
except Exception:
    stop_words_ar = set()

# Darija stopwords depuis un fichier .txt
darija_path = "../../data/processed/darija_stopwords.txt"
darija_stopwords = set()
try:
    with open(darija_path, "r", encoding="utf-8") as f:
        darija_stopwords = {line.strip() for line in f if line.strip()}
except FileNotFoundError:
    darija_stopwords = set()

# Fusion arabe + darija
stop_words_ar = stop_words_ar.union(darija_stopwords)
stop_words_ar.add("و")
lemmatizer = WordNetLemmatizer()

arabic_diacritics = re.compile(r'[\u064B-\u065F\u0670\u0640]')

def _normalize_basic(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _normalize_ar(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = arabic_diacritics.sub("", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_classic(text: str, lang: str) -> str:
    lang = (lang or 'ENG').upper()
    if lang == 'DAR':
        text = _normalize_ar(text)
        tokens = [tok for tok in text.split() if tok and tok not in stop_words_ar]
        return " ".join(tokens)
    if lang == 'FR':
        text = _normalize_basic(text)
        tokens = [tok for tok in word_tokenize(text) if tok.isalpha() and tok not in stop_words_fr]
        return " ".join(tokens)
    # Default: ENG
    text = _normalize_basic(text)
    tokens = [tok for tok in word_tokenize(text) if tok.isalpha() and tok not in stop_words_en]
    lemmas = [lemmatizer.lemmatize(tok) for tok in tokens]
    return " ".join(lemmas)

def preprocess_light(text: str, lang: str) -> str:
    lang = (lang or 'ENG').upper()
    if lang == 'DAR':
        return _normalize_ar(text)
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print('Preprocessing functions ready.')

Preprocessing functions ready.


In [5]:
print('Running classic preprocessing...')
df_multi['text_clean_classique'] = df_multi.apply(
    lambda row: preprocess_classic(row['text'], row['language']), axis=1
)
print('Running light preprocessing...')
df_multi['text_clean_light'] = df_multi.apply(
    lambda row: preprocess_light(row['text'], row['language']), axis=1
)
print(df_multi[['text', 'language', 'text_clean_classique', 'text_clean_light']].head(5).to_markdown(index=False))

Running classic preprocessing...
Running light preprocessing...
| text                                                                     | language   | text_clean_classique                              | text_clean_light                                                         |
|:-------------------------------------------------------------------------|:-----------|:--------------------------------------------------|:-------------------------------------------------------------------------|
| لماذا تتوقف المقاومة وتتراجع أثناء الطهي؟ أجهزة الأمن؟ التدفئة؟ أين خطأ؟ | ARABIC     | لماذا تتوقف المقاومة وتتراجع أثناء أجهزة أين      | لماذا تتوقف المقاومة وتتراجع أثناء الطهي؟ أجهزة الأمن؟ التدفئة؟ أين خطأ؟ |
| ويتعين عليّ ربط جهاز استقبال خارجي                                        | ARABIC     | ويتعين ربط جهاز استقبال خارجي                     | ويتعين عليّ ربط جهاز استقبال خارجي                                        |
| منتج جيد جداً ، مرضي جداً                                       

In [6]:
df_multi.to_csv("../../data/processed/multilingual_clean.csv", index=False)
print('Saved to ../../data/processed/multilingual_clean.csv')   

Saved to ../../data/processed/multilingual_clean.csv
